In [5]:
import joblib
import pandas as pd
from pythainlp.tokenize import word_tokenize
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# 1. โหลดข้อมูลไฟล์ CSV ภายนอกทั้งหมด
file_list = [
    "AFNC_Opendata_export_20260930221100(2025).csv",
    "AFNC_Opendata_export_20260930221100(2026).csv"
]
df_external = pd.concat([pd.read_csv(f, encoding='utf-8-sig') for f in file_list], ignore_index=True)

total_news = len(df_external)
print(f"จำนวนข่าว: {total_news}")

y_true = []
y_pred = []

# โหลดโมเดล SVM และ Vectorizer จากไฟล์ใน Colab
vectorizer = joblib.load('tfidf_vectorizer.pkl')
model = joblib.load('fake_news_svm_model.pkl')

# 2. วนลูปประมวลผลข้อมูลทั้งหมดและแสดงความคืบหน้า
batch_size = 50
for i in range(0, total_news, batch_size):
    end_idx = min(i + batch_size, total_news)
    batch_df = df_external.iloc[i : end_idx]

    # คอลัมน์ D = เนื้อหาข่าว (index 3), คอลัมน์ F = ประเภทข่าว (index 5)
    X_text = batch_df.iloc[:, 3].astype(str)
    batch_true = batch_df.iloc[:, 5].apply(lambda x: 1 if 'ข่าวจริง' in str(x).strip() else 0).tolist()

    X_vec = vectorizer.transform(X_text)
    batch_pred = model.predict(X_vec).tolist()

    y_true.extend(batch_true)
    y_pred.extend(batch_pred)

    print(f"{end_idx}/{total_news}")

print("ตอบไม่ได้: 0\n")

# 3. คำนวณและแสดงผลลัพธ์ในรูปแบบเดียวกับ Typhoon LLM
svm_acc = accuracy_score(y_true, y_pred)
svm_prec, svm_rec, svm_f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary', zero_division=0)

print("External: TF-IDF + SVM")
print("==================================================")
print(f"Accuracy  : {svm_acc:.4f}")
print(f"Precision : {svm_prec:.4f}")
print(f"Recall    : {svm_rec:.4f}")
print(f"F1-score  : {svm_f1:.4f}")

จำนวนข่าว: 259
50/259
100/259
150/259
200/259
250/259
259/259
ตอบไม่ได้: 0

External: TF-IDF + SVM
Accuracy  : 0.1622
Precision : 0.1513
Recall    : 0.2072
F1-score  : 0.1749


In [9]:
!huggingface-cli login


Hint: A new version of huggingface_hub (2.1.1) is available! You are using version 1.31.0.
To update, run: hf update
Hint: `hf` is already installed! Use it directly.

Hint: Examples:
  hf auth login
  hf download unsloth/gemma-4-31B-it-GGUF
  hf upload my-cool-model . .
  hf models ls --search "gemma"
  hf repos ls --format json
  hf jobs run python:3.12 python -c 'print("Hello!")'
  hf --help



In [12]:
from huggingface_hub import login
login("")

In [14]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# ใช้ชื่อรีโปฯ ที่ถูกต้องบน Hugging Face
model_name = "airesearch/wangchanberta-base-att-spm-uncased"
print(f"กำลังโหลดโมเดล: {model_name}")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
model.eval()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

total_news = len(df_external)
y_true_bert = []
y_pred_bert = []

batch_size = 32
for i in range(0, total_news, batch_size):
    end_idx = min(i + batch_size, total_news)
    batch_df = df_external.iloc[i : end_idx]

    # ใช้ชุดข้อมูล df_external ตัวเดิมต่อจาก SVM
    X_text = batch_df.iloc[:, 5].astype(str).tolist()
    batch_true = batch_df.iloc[:, 3].apply(lambda x: 1 if 'ข่าวจริง' in str(x).strip() else 0).tolist()

    inputs = tokenizer(X_text, padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        predictions = torch.argmax(outputs.logits, dim=-1).cpu().tolist()

    y_true_bert.extend(batch_true)
    y_pred_bert.extend(predictions)

    print(f"{end_idx}/{total_news}")

print("ตอบไม่ได้: 0\n")

# คำนวณและแสดงผลลัพธ์
bert_acc = accuracy_score(y_true_bert, y_pred_bert)
bert_prec, bert_rec, bert_f1, _ = precision_recall_fscore_support(y_true_bert, y_pred_bert, average='binary', zero_division=0)

print("External: Thai BERT")
print("==================================================")
print(f"Accuracy  : {bert_acc:.4f}")
print(f"Precision : {bert_prec:.4f}")
print(f"Recall    : {bert_rec:.4f}")
print(f"F1-score  : {bert_f1:.4f}")

กำลังโหลดโมเดล: airesearch/wangchanberta-base-att-spm-uncased


config.json:   0%|          | 0.00/546 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/282 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/905k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  423MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] CamembertForSequenceClassification LOAD REPORT from: airesearch/wangchanberta-base-att-spm-uncased
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


32/259
64/259
96/259
128/259
160/259
192/259
224/259
256/259
259/259
ตอบไม่ได้: 0

External: Thai BERT
Accuracy  : 0.1274
Precision : 0.1274
Recall    : 1.0000
F1-score  : 0.2260


In [15]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

total_news = len(df_external)
y_true_typhoon = []
y_pred_typhoon = []
not_answered = 0

print(f"จำนวนข่าวทั้งหมด: {total_news}")

# ลูปประมวลผลผ่านชุดข้อมูลภายนอก
for i in range(0, total_news):
    row = df_external.iloc[i]
    text = str(row.iloc[5]) # คอลัมน์เนื้อหาข่าว
    true_label = 1 if 'ข่าวจริง' in str(row.iloc[3]).strip() else 0 # ป้ายกำกับจริง

    y_true_typhoon.append(true_label)

    # 💡 [ส่วนเรียกโมเดล Typhoon จากในไฟล์ .ipynb ของคุณ]
    # แทนค่าผลลัพธ์ที่ได้จาก Typhoon ตรงนี้ (0 = ข่าวปลอม, 1 = ข่าวจริง, ถ้าตอบไม่ได้ให้ข้ามหรือนับ not_answered += 1)
    pred_label = 0 # ตัวอย่างค่าทำนาย
    y_pred_typhoon.append(pred_label)

    if (i + 1) % 32 == 0 or (i + 1) == total_news:
        print(f"{i + 1}/{total_news}")

print(f"ตอบไม่ได้: {not_answered}\n")

# คำนวณและแสดงผลลัพธ์ในรูปแบบมาตรฐาน
typhoon_acc = accuracy_score(y_true_typhoon, y_pred_typhoon)
typhoon_prec, typhoon_rec, typhoon_f1, _ = precision_recall_fscore_support(y_true_typhoon, y_pred_typhoon, average='binary', zero_division=0)

print("External: Typhoon")
print("==================================================")
print(f"Accuracy  : {typhoon_acc:.4f}")
print(f"Precision : {typhoon_prec:.4f}")
print(f"Recall    : {typhoon_rec:.4f}")
print(f"F1-score  : {typhoon_f1:.4f}")

จำนวนข่าวทั้งหมด: 259
32/259
64/259
96/259
128/259
160/259
192/259
224/259
256/259
259/259
ตอบไม่ได้: 0

External: Typhoon
Accuracy  : 0.8726
Precision : 0.0000
Recall    : 0.0000
F1-score  : 0.0000


In [16]:
import pandas as pd

# รวบรวมผลลัพธ์ของแต่ละโมเดลเข้าด้วยกัน
comparison_data = [
    {
        "Model": "TF-IDF + SVM",
        "Internal Acc": 0.8313, # ใส่ค่า Internal ของ SVM (หรือดึงจากตัวแปรของคุณ)
        "Internal F1": 0.8214,
        "External Acc": svm_acc,
        "External Prec": svm_prec,
        "External Recall": svm_rec,
        "External F1": svm_f1,
        "F1 Drop": 0.8214 - svm_f1 # ตัวอย่างการคำนวณส่วนต่าง F1
    },
    {
        "Model": "Thai BERT",
        "Internal Acc": 0.8243,
        "Internal F1": 0.8279,
        "External Acc": bert_acc,
        "External Prec": bert_prec,
        "External Recall": bert_rec,
        "External F1": bert_f1,
        "F1 Drop": 0.8279 - bert_f1
    },
    {
        "Model": "Typhoon LLM",
        "Internal Acc": 0.7667,
        "Internal F1": 0.7518,
        "External Acc": typhoon_acc,
        "External Prec": typhoon_prec,
        "External Recall": typhoon_rec,
        "External F1": typhoon_f1,
        "F1 Drop": 0.7518 - typhoon_f1
    }
]

# แปลงเป็น DataFrame เพื่อแสดงตาราง
df_summary = pd.DataFrame(comparison_data)

# จัดรูปแบบทศนิยม 4 ตำแหน่งให้อ่านง่าย
df_styled = df_summary.style.format({
    "Internal Acc": "{:.4f}",
    "Internal F1": "{:.4f}",
    "External Acc": "{:.4f}",
    "External Prec": "{:.4f}",
    "External Recall": "{:.4f}",
    "External F1": "{:.4f}",
    "F1 Drop": "{:.4f}"
})

df_styled

,Model,Internal Acc,Internal F1,External Acc,External Prec,External Recall,External F1,F1 Drop
0,TF-IDF + SVM,0.8313,0.8214,0.1622,0.1513,0.2072,0.1749,0.6465
1,Thai BERT,0.8243,0.8279,0.1274,0.1274,1.0000,0.2260,0.6019
2,Typhoon LLM,0.7667,0.7518,0.8726,0.0000,0.0000,0.0000,0.7518
